# Índice de evidência — achar no cru o bloco que você gostou no derivado

**O que este notebook é.** Um índice, nada mais: você está em `derivados/<caso>.json`, achou um bloco
interessante, e quer ir direto naquele mesmo lugar em `crus/<exec_id>.json` — o trace cru, sem nenhuma
alteração. Este notebook faz só isso: lista casos, lista os blocos de um caso, e busca qualquer caminho
no cru. Não interpreta nada, não verifica nada em massa, não é pipeline — é uma lanterna.

**Um notebook só, para qualquer pasta.** Todas as pastas de `resultados/evidencia/` têm o mesmo formato
(é assim de propósito — ver `../docs/03-procedimento-validacao.md`, "Evidência por análise"), então o
mesmo notebook serve para todas: você só troca a pasta e o caso nas células abaixo.

**Como abrir:** a partir da raiz do repo, `uv run jupyter lab` e abra este arquivo — mesmo kernel do
notebook principal. Não precisa de pandas/numpy, só `json` da biblioteca padrão.

> ⚠️ **Nunca salvar/commitar este notebook com saídas.** Diferente do notebook principal da análise
> (que só mostra estrutura redigida), rodar este aqui imprime texto **cru** do trace — pode ter nome de
> cliente, número de processo, conteúdo de documento. Antes de salvar ou dar `git add`, limpar as saídas:
> `Kernel > Restart Kernel and Clear Outputs...` no Jupyter, ou
> `uv run jupyter nbconvert --clear-output --inplace pipeline/indice_evidencia.ipynb` no terminal.

In [ ]:
import glob, json, os

EVIDENCIA = "resultados/evidencia"

def carregar_casos(pasta):
    import csv
    with open(f"{EVIDENCIA}/{pasta}/casos.csv", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def resolver_exec(pasta, prefixo):
    """Aceita o exec_id inteiro ou só os 8 primeiros caracteres."""
    achados = glob.glob(f"{EVIDENCIA}/{pasta}/crus/{prefixo}*.json")
    if len(achados) != 1:
        raise ValueError(f"'{prefixo}' achou {len(achados)} cru(s) em {pasta}/crus/: {achados}")
    return os.path.basename(achados[0])[:-5]

def carregar_cru(pasta, exec_prefixo):
    exec_id = resolver_exec(pasta, exec_prefixo)
    with open(f"{EVIDENCIA}/{pasta}/crus/{exec_id}.json", encoding="utf-8") as f:
        return json.load(f)

def carregar_derivado(pasta, exec_prefixo, role, idx):
    exec_id = resolver_exec(pasta, exec_prefixo)
    with open(f"{EVIDENCIA}/{pasta}/derivados/{exec_id[:8]}_{role}_idx{idx}.json", encoding="utf-8") as f:
        return json.load(f)

def seguir_caminho(obj, caminho):
    """A função inteira: entra em cada chave/índice do caminho, na ordem. Nenhuma outra lógica."""
    for chave in caminho:
        obj = obj[chave]
    return obj

## 1 · Escolha a pasta e veja os casos disponíveis

Troque `PASTA` abaixo por qualquer uma de `resultados/evidencia/` (ex.: `11.4_conserto`,
`11.7_amostra_passo6`) e rode a célula.

In [ ]:
PASTA = "11.7_amostra_passo6"

for i, c in enumerate(carregar_casos(PASTA), 1):
    print(f"{i:2d}. exec={c['exec_id'][:8]}  role={c['role']}  idx={c['idx']}  "
          f"mes={c.get('mes', '?')}  motivo={c.get('motivo', '')}")

## 2 · Escolha um caso e veja os blocos disponíveis (sem mostrar texto ainda)

Por padrão a célula pega o 1º caso da lista acima; para outro, preencha `EXEC`/`ROLE`/`IDX` com um dos casos acima (`EXEC` aceita só os 8 primeiros caracteres; `IDX` é
a coluna `idx` da tabela, não o `step_number`).

In [ ]:
# 1º caso da pasta por padrão (os casos mudam quando a pasta é regenerada — não fixar um exec_id aqui);
# para outro caso: EXEC, ROLE, IDX = "<8 primeiros do exec_id>", "<role>", <idx>
c0 = carregar_casos(PASTA)[0]
EXEC, ROLE, IDX = c0["exec_id"][:8], c0["role"], int(c0["idx"])

derivado = carregar_derivado(PASTA, EXEC, ROLE, IDX)
for i, t in enumerate(derivado["trechos_do_cru"]):
    extra = f"  caracteres={t['caracteres']}" if "caracteres" in t else (f"  mensagens={t['mensagens']}" if "mensagens" in t else "")
    print(f"{i}. {t['o_que']}{extra}\n   cru: {t['cru']}")

## 3 · Achou um bloco que gostou? Veja o caminho dele e o texto que o derivado guardou

Troque `N` pelo número do bloco (a lista acima). O aviso de PII é sério — pode ter nome de cliente, número
de processo ou texto de documento; não copie a saída desta célula para fora deste notebook.

In [ ]:
N = 0   # troque pelo número do bloco na lista acima

t = derivado["trechos_do_cru"][N]
print("o que é:", t["o_que"])
print("caminho no cru:", t["cru"])
print("\n# ATENÇÃO: pode conter nome de cliente, número de processo ou texto de documento.\n")
print(t["texto"])

## 4 · Vá direto naquele caminho no cru — sem passar pelo derivado

É o índice em si: pega o caminho do bloco 3 (`t["cru"]`) e busca **de novo**, direto no arquivo cru, com
a função mais simples que existe (`seguir_caminho`, célula 1) — sem nenhum código de análise no meio.

In [ ]:
cru = carregar_cru(PASTA, EXEC)
seguir_caminho(cru, t["cru"])

### Bateu?

Se o que saiu na célula 4 for igual ao texto da célula 3, o bloco que você achou no derivado é fiel ao
cru — você conferiu com seus próprios olhos, sem depender do código que gerou o derivado.

### Caminho livre, sem passar por um bloco pronto

Você também pode ir a **qualquer** lugar do cru, mesmo que não exista um bloco pronto pra ele — é só montar
a lista de chaves/índices você mesmo. `'txt_etap_memo'` → o nome do papel → o índice do step na lista
completa (`posicao_no_cru.indice_na_lista_do_papel` no arquivo derivado) → o nome do campo que quiser
(`code_action`, `model_output`, `observations`, `error`…).

## 5 · Ver o system prompt inteiro do step (não só o bloco da ferramenta)

Os blocos de `trechos_do_cru` só trazem o pedaço em que a ferramenta escolhida é declarada, mais outras
linhas que a citam — não o prompt inteiro, pra não inflar cada arquivo derivado com texto repetido. Mas o
caminho pra chegar nele é conhecido: é a primeira mensagem do contexto daquele mesmo step. Usa o índice
bruto que já está em `posicao_no_cru` do derivado — não precisa descobrir esse número de novo.

In [ ]:
sysprompt_inteiro = seguir_caminho(cru, [
    "txt_etap_memo", ROLE, derivado["posicao_no_cru"]["indice_na_lista_do_papel"],
    "model_input_messages", 0, "content", 0, "text",
])

import re
print("tamanho:", len(sysprompt_inteiro), "caracteres")
print("ferramentas declaradas neste prompt:", re.findall(r"^def (\w+)\(", sysprompt_inteiro, re.M))

Se quiser ler o texto inteiro (aviso de PII, igual à célula 3):

In [ ]:
print("# ATENÇÃO: pode conter nome de cliente, número de processo ou texto de documento.\n")
print(sysprompt_inteiro)

In [ ]:
# a mensagem de erro do próprio step do caso (índice bruto na lista do papel, lido do derivado)
seguir_caminho(cru, ["txt_etap_memo", ROLE, derivado["posicao_no_cru"]["indice_na_lista_do_papel"], "error", "message"])